In [3]:

# 0) Installs (idempotent)
!pip install -q transformers accelerate bitsandbytes seaborn scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 43.4 MB/s eta 0:00:00


In [2]:
# ============================================================
# PropInsight — Setup + Data Prep + Baseline (AISG SEA-LION 32B IT)
# ============================================================


# 1) Imports
import os, json, re, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, torch
from pathlib import Path
from tqdm.auto import tqdm
from collections import Counter
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import drive

# 2) Mount Drive
drive.mount('/content/drive')

# 3) Directory configuration (Path-only, no duplicates)
BASE = Path("/content/drive/MyDrive/PropInsight")

# Primary folders
CORPUS_DIR      = BASE / "corpus"
LABELED_DIR     = BASE / "labeled"
RAW_DIR         = BASE / "raw"
RESI_DIR        = BASE / "RESI"
PROCESSED_DIR   = BASE / "processed"
PREPROCESS_DIR  = BASE / "preprocess"
ANALYTICS_DIR   = BASE / "analytics"
CACHE_DIR       = BASE / "cache"

# Generated training files
DATASETS_DIR    = BASE / "propinsight_datasets"
TRAIN_JSON      = DATASETS_DIR / "train.json"
VAL_JSON        = DATASETS_DIR / "validation.json"
TEST_JSON       = DATASETS_DIR / "test.json"
META_JSON       = DATASETS_DIR / "dataset_metadata.json"
LABEL_MAPS      = DATASETS_DIR / "label_maps.json"
CLASS_WEIGHTS   = DATASETS_DIR / "class_weights.json"

# Model + results + visualizations
MODELS_DIR          = BASE / "models"
FINETUNED_DIR       = MODELS_DIR / "finetuned"
RESULTS_DIR         = BASE / "results"
VISUALIZATIONS_DIR  = BASE / "visualizations"

# Output files
TRAINING_CONFIG       = RESULTS_DIR / "training_config.json"
EVAL_RESULTS          = RESULTS_DIR / "evaluation_results.json"
TRAINING_CURVES       = RESULTS_DIR / "training_curves.png"
EVAL_PLOTS            = RESULTS_DIR / "evaluation_plots.png"
EVAL_REPORT_TXT       = RESULTS_DIR / "evaluation_report.txt"
BASELINE_PREDICTIONS  = RESULTS_DIR / "baseline_predictions.json"
FINETUNED_PREDICTIONS = RESULTS_DIR / "finetuned_predictions.json"
COMPREHENSIVE_METRICS = RESULTS_DIR / "comprehensive_metrics.json"
EVAL_REPORT_JSON      = RESULTS_DIR / "evaluation_report.json"
RESI_ALIGNMENT_JSON   = RESULTS_DIR / "resi_alignment.json"

# Logs
TENSORBOARD_LOGS = RESULTS_DIR / "tensorboard"
TRAINING_LOG     = RESULTS_DIR / "qwen_sealion_finetune.log"

# Evaluation outputs
EVAL_OUT_DIR            = BASE / "evaluation_results"
STANDALONE_EVAL_RESULTS = EVAL_OUT_DIR / "evaluation_results.json"

# Ensure dirs
for d in [
    BASE, CORPUS_DIR, LABELED_DIR, RAW_DIR, RESI_DIR, PROCESSED_DIR,
    PREPROCESS_DIR, ANALYTICS_DIR, CACHE_DIR, DATASETS_DIR, MODELS_DIR,
    FINETUNED_DIR, RESULTS_DIR, VISUALIZATIONS_DIR, TENSORBOARD_LOGS,
    EVAL_OUT_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

print("✅ Directory configuration complete.")
print(f"Base: {BASE}")

# 4) Data Prep (creates train/validation/test once)
def load_csv(p: Path):
    return pd.read_csv(p) if p.exists() else pd.DataFrame()

dfs = [
    load_csv(LABELED_DIR/"forums/singapore_property_forum_posts_sgexpats_processed/sgexpats_forum_labeled.csv"),
    load_csv(LABELED_DIR/"multi_forum_property_posts_hwz_processed_2023_2025/forum_labeled_corrected.csv"),
    load_csv(LABELED_DIR/"government/gov_websites_labeled.csv"),
    load_csv(LABELED_DIR/"reddit/reddit_2023_2025_property_labeled.csv"),
]
merged = pd.concat([d for d in dfs if not d.empty], ignore_index=True).fillna("")
print("Merged rows:", len(merged))

def pick(row, *keys, default=""):
    for k in keys:
        if k in row and str(row[k]).strip() != "":
            return row[k]
    return default

records=[]
for _, r in merged.iterrows():
    text = pick(r,"comment","post_text","body","comment_body","text","clean_text")
    if not isinstance(text,str) or len(text.strip()) < 10:
        continue
    records.append({
        "instruction":"Analyze the sentiment of this Singapore property comment:",
        "input": text.strip(),
        "output": f"""Overall Sentiment: {pick(r,'overall_sentiment','Sentiment','neutral')}
Price Sentiment: {pick(r,'price_sentiment','PriceSentiment','neutral')}
Policy Sentiment: {pick(r,'policy_sentiment','PolicySentiment','neutral')}
Affordability Sentiment: {pick(r,'affordability_sentiment','AffordabilitySentiment','neutral')}
Location: {pick(r,'location','Location','Not specified')}
Aspect: {pick(r,'aspect','Aspect','general')}
Entity: {pick(r,'entities','Entity','None')}
Policy Mentioned: {pick(r,'policy_mentioned','PolicyMentioned','None')}
Singlish Detected: {'Yes' if bool(pick(r,'singlish_detected','SinglishDetected','has_singlish',False)) else 'No'}
Cultural Context: {pick(r,'cultural_context','CulturalContext','None')}
Emotion: {pick(r,'emotion','Emotion','neutral')}
Datetime: {pick(r,'date','Datetime','datetime','')}
Source: {pick(r,'source','Source','subreddit','forum_name','unknown')}
Reasoning: Analysis of the sentiment and context based on Singapore property market dynamics.""",
    })

print("Prepared:", len(records))
if len(records) == 0:
    raise RuntimeError("No usable rows; check your labeled CSVs under /labeled/")

df = pd.DataFrame(records)
train_idx, test_idx = train_test_split(df.index, test_size=0.15, random_state=42)
train_idx, val_idx  = train_test_split(train_idx, test_size=0.10, random_state=42)

train = df.loc[train_idx].to_dict(orient="records")
val   = df.loc[val_idx].to_dict(orient="records")
test  = df.loc[test_idx].to_dict(orient="records")

json.dump(train, open(TRAIN_JSON, "w"), indent=2, ensure_ascii=False)
json.dump(val,   open(VAL_JSON,   "w"), indent=2, ensure_ascii=False)
json.dump(test,  open(TEST_JSON,  "w"), indent=2, ensure_ascii=False)
print("✓ Saved splits to:", DATASETS_DIR)

# 5) Baseline Inference (AISG SEA-LION 32B IT; no LoRA)
test_data = json.load(open(TEST_JSON, "r", encoding="utf-8"))
print(f"✓ Loaded test set: {len(test_data)} rows")

MODEL_ID = "aisingapore/Qwen-SEA-LION-v4-32B-IT"  # AISG base IT model

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# NOTE: Transformers expects 'torch_dtype'. We set a variable named 'dtype' and pass it to 'torch_dtype'.
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=dtype,
)
model.eval()
print("✓ Baseline model loaded")

SYSTEM_MSG = (
    "You are PropInsight, a Singapore real-estate expert. "
    "Return ONLY the Output block with fields: Overall Sentiment, Price Sentiment, "
    "Policy Sentiment, Affordability Sentiment, Location, Aspect, Entity, Policy Mentioned, "
    "Singlish Detected, Cultural Context, Emotion, Datetime, Source, Reasoning."
)

def build_prompt(comment_text: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user",
         "content": f"Analyze the sentiment of this Singapore property comment:\n\n{comment_text}\n\nReturn only the Output block."},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def extract_field(block: str, key: str, default: str = "") -> str:
    m = re.search(rf"{re.escape(key)}\s*:\s*(.*)", block, flags=re.IGNORECASE)
    return (m.group(1).strip() if m else default)

predictions = []
for item in tqdm(test_data, total=len(test_data), desc="Baseline inference"):
    inp = item.get("input", "")
    if not isinstance(inp, str) or len(inp.strip()) < 1:
        predictions.append({"input": inp,"ground_truth": item.get("output",""),"prediction": "","metadata": item.get("metadata", {})})
        continue

    prompt = build_prompt(inp)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
        )
    text = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

    predictions.append({
        "input": inp,
        "ground_truth": item.get("output",""),
        "prediction": text,
        "metadata": item.get("metadata", {})
    })

with open(BASELINE_PREDICTIONS, "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=2, ensure_ascii=False)
print(f"✓ Saved predictions → {BASELINE_PREDICTIONS}")

# Quick metrics + plots (Overall Sentiment)
def pick_overall(block: str) -> str:
    val = extract_field(block, "Overall Sentiment", default="neutral").lower()
    return val if val else "neutral"

y_true = [pick_overall(p["ground_truth"]) for p in predictions]
y_pred = [pick_overall(p["prediction"]) for p in predictions]
gt_counts = Counter(y_true); pd_counts = Counter(y_pred)
labels = sorted(set(list(gt_counts.keys()) + list(pd_counts.keys())))
overall_acc = np.mean([t == p for t, p in zip(y_true, y_pred)]) if len(y_true) else 0.0

metrics = {
    "samples": len(predictions),
    "overall_sentiment_accuracy": overall_acc,
    "label_set": labels,
    "ground_truth_distribution": gt_counts,
    "prediction_distribution": pd_counts,
}
with open(RESULTS_DIR / "baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, default=lambda x: dict(x))
print(f"✓ Overall Sentiment accuracy: {overall_acc:.3f}")

sns.set_style("whitegrid")
x = np.arange(len(labels))
gt_vals = [gt_counts.get(l,0) for l in labels]
pd_vals = [pd_counts.get(l,0) for l in labels]
fig, ax = plt.subplots(figsize=(11,5))
ax.bar(x-0.35/2, gt_vals, width=0.35, label="Ground Truth")
ax.bar(x+0.35/2, pd_vals, width=0.35, label="Predicted")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=0)
ax.set_title("Overall Sentiment — Baseline (GT vs Pred)")
ax.legend(); ax.grid(axis="y", alpha=.3)
plt.tight_layout()
plt.savefig(VISUALIZATIONS_DIR / "sentiment_comparison.png", dpi=300)
plt.close()
print(f"✓ Plot saved → {VISUALIZATIONS_DIR / 'sentiment_comparison.png'}")

from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_true, y_pred, labels=labels)
with np.errstate(invalid='ignore'):
    cmn = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=labels, yticklabels=labels, ax=ax,
            cbar_kws={"label":"Proportion"})
ax.set_xlabel("Predicted"); ax.set_ylabel("Ground Truth")
ax.set_title("Overall Sentiment — Baseline Confusion Matrix")
plt.tight_layout()
plt.savefig(VISUALIZATIONS_DIR / "confusion_matrix.png", dpi=300)
plt.close()
print(f"✓ Plot saved → {VISUALIZATIONS_DIR / 'confusion_matrix.png'}")

print("✅ BASELINE COMPLETE")


KeyboardInterrupt: 

In [3]:
# ============================================================
# PropInsight — Baseline SEA-LION Comprehensive Evaluation
# Copy this entire cell and run it in Google Colab
# ============================================================

import os
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
from collections import Counter
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

print("="*70)
print("PropInsight - Baseline SEA-LION Comprehensive Evaluation")
print("="*70 + "\n")

# ========== CONFIGURATION ==========
BASE = Path("/content/drive/MyDrive/PropInsight")
RESULTS_DIR = BASE / "results"
VISUALIZATIONS_DIR = BASE / "visualizations"
BASELINE_PREDICTIONS = RESULTS_DIR / "baseline_predictions.json"

# Output files with baseline_Sealion suffix
BASELINE_METRICS_JSON = RESULTS_DIR / "baseline_Sealion_metrics.json"
BASELINE_REPORT_JSON = RESULTS_DIR / "baseline_Sealion_evaluation_report.json"
BASELINE_REPORT_TXT = RESULTS_DIR / "baseline_Sealion_evaluation_report.txt"

# Visualization outputs
VIS_NLP_METRICS = VISUALIZATIONS_DIR / "baseline_Sealion_nlp_metrics.png"
VIS_SENTIMENT_COMPARISON = VISUALIZATIONS_DIR / "baseline_Sealion_sentiment_comparison.png"
VIS_CONFUSION_MATRIX = VISUALIZATIONS_DIR / "baseline_Sealion_confusion_matrix.png"
VIS_PER_LABEL_F1 = VISUALIZATIONS_DIR / "baseline_Sealion_per_label_f1.png"
VIS_COMPREHENSIVE = VISUALIZATIONS_DIR / "baseline_Sealion_comprehensive_metrics.png"

# Ensure directories exist
for d in [RESULTS_DIR, VISUALIZATIONS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"✓ Base directory: {BASE}")
print(f"✓ Loading predictions from: {BASELINE_PREDICTIONS.name}\n")


PropInsight - Baseline SEA-LION Comprehensive Evaluation

✓ Base directory: /content/drive/MyDrive/PropInsight
✓ Loading predictions from: baseline_predictions.json



In [4]:
# ========== HELPER FUNCTIONS ==========
def extract_field(block: str, key: str, default: str = "") -> str:
    """Extract a field value from the output block."""
    if not isinstance(block, str):
        return default
    m = re.search(rf"{re.escape(key)}\s*:\s*(.*?)(?:\n|$)", block, flags=re.IGNORECASE)
    return (m.group(1).strip() if m else default)


def normalize_sentiment(sentiment: str) -> str:
    """Normalize sentiment labels to lowercase and handle variations."""
    if not isinstance(sentiment, str):
        return "neutral"

    sentiment = sentiment.lower().strip()

    # Handle common variations
    if sentiment in ["positive", "pos"]:
        return "positive"
    elif sentiment in ["negative", "neg"]:
        return "negative"
    elif sentiment in ["neutral", "neu", "mixed"]:
        return "neutral"
    else:
        return sentiment if sentiment else "neutral"


In [6]:
# ========== LOAD PREDICTIONS ==========
print("="*70)
print("STEP 1: LOADING BASELINE PREDICTIONS")
print("="*70)

import json
from pathlib import Path
from google.colab import files

BASE = Path("/content/drive/MyDrive/PropInsight")
RESULTS_DIR = BASE / "results"
BASELINE_PREDICTIONS = RESULTS_DIR / "baseline_predictions.json"

# Check if the file exists, if not, prompt user to upload
if not BASELINE_PREDICTIONS.exists():
    print(f"❌ File not found: {BASELINE_PREDICTIONS}")
    print("Please upload the baseline_predictions.json file.")
    uploaded = files.upload()

    if BASELINE_PREDICTIONS.name in uploaded:
        # Save the uploaded file to the correct location
        try:
            with open(BASELINE_PREDICTIONS, 'wb') as f:
                f.write(uploaded[BASELINE_PREDICTIONS.name])
            print(f"✓ Successfully uploaded and saved {BASELINE_PREDICTIONS.name} to {BASELINE_PREDICTIONS}")
        except Exception as e:
            print(f"❌ Error saving uploaded file: {e}")
            # If saving fails, the file is still in the /content directory temporarily
            # You might want to add instructions here on how to move it manually
            print(f"Uploaded file is temporarily available in the current directory.")
            raise # Re-raise the exception after printing
    else:
        print("❌ Upload cancelled or incorrect file name.")
        raise FileNotFoundError(f"Prediction file not found after upload attempt: {BASELINE_PREDICTIONS}")

try:
    with open(BASELINE_PREDICTIONS, "r", encoding="utf-8") as f:
        baseline_preds = json.load(f)
    print(f"✓ Successfully loaded {len(baseline_preds)} predictions\n")
except FileNotFoundError:
    # This case should ideally not be reached if the upload or initial check worked
    print(f"❌ ERROR: File not found after checks: {BASELINE_PREDICTIONS}")
    raise
except Exception as e:
    print(f"❌ ERROR loading predictions: {e}")
    raise


# ========== EXTRACT OVERALL SENTIMENT ==========
print("="*70)
print("STEP 2: EXTRACTING OVERALL SENTIMENT")
print("="*70)

# Assuming baseline_preds is loaded successfully
y_true = []
y_pred = []

for item in baseline_preds:
    gt = extract_field(item.get("ground_truth", ""), "Overall Sentiment", "neutral")
    pred = extract_field(item.get("prediction", ""), "Overall Sentiment", "neutral")

    y_true.append(normalize_sentiment(gt))
    y_pred.append(normalize_sentiment(pred))

print(f"✓ Extracted {len(y_true)} ground truth labels")
print(f"✓ Extracted {len(y_pred)} predictions")

# Get all unique labels
labels = sorted(list(set(y_true) | set(y_pred)))
print(f"✓ Found {len(labels)} unique sentiment labels: {labels}\n")

STEP 1: LOADING BASELINE PREDICTIONS
❌ File not found: /content/drive/MyDrive/PropInsight/results/baseline_predictions.json
Please upload the baseline_predictions.json file.


Saving baseline_predictions.json to baseline_predictions.json
✓ Successfully uploaded and saved baseline_predictions.json to /content/drive/MyDrive/PropInsight/results/baseline_predictions.json
✓ Successfully loaded 554 predictions

STEP 2: EXTRACTING OVERALL SENTIMENT
✓ Extracted 554 ground truth labels
✓ Extracted 554 predictions
✓ Found 3 unique sentiment labels: ['negative', 'neutral', 'positive']



In [7]:




# ========== CALCULATE STANDARD METRICS ==========
print("="*70)
print("STEP 3: CALCULATING STANDARD METRICS")
print("="*70)

baseline_metrics = {
    "model": "AISG SEA-LION v4 32B IT (Baseline)",
    "total_samples": len(baseline_preds),
    "labels": labels,
    "overall_sentiment_accuracy": float(accuracy_score(y_true, y_pred)),
    "macro_precision": float(precision_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)),
    "macro_recall": float(recall_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)),
    "macro_f1": float(f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)),
    "weighted_precision": float(precision_score(y_true, y_pred, labels=labels, average='weighted', zero_division=0)),
    "weighted_recall": float(recall_score(y_true, y_pred, labels=labels, average='weighted', zero_division=0)),
    "weighted_f1": float(f1_score(y_true, y_pred, labels=labels, average='weighted', zero_division=0)),
}

print(f"✓ Overall Accuracy:     {baseline_metrics['overall_sentiment_accuracy']:.4f}")
print(f"✓ Macro Precision:      {baseline_metrics['macro_precision']:.4f}")
print(f"✓ Macro Recall:         {baseline_metrics['macro_recall']:.4f}")
print(f"✓ Macro F1:             {baseline_metrics['macro_f1']:.4f}")
print(f"✓ Weighted F1:          {baseline_metrics['weighted_f1']:.4f}\n")


# ========== CALCULATE PER-LABEL METRICS ==========
print("="*70)
print("STEP 4: CALCULATING PER-LABEL METRICS")
print("="*70)

per_label_f1 = {}
per_label_precision = {}
per_label_recall = {}
per_label_support = {}

for label in labels:
    # Binary classification for each label
    y_true_label = [1 if yt == label else 0 for yt in y_true]
    y_pred_label = [1 if yp == label else 0 for yp in y_pred]

    # Count support (number of true instances)
    support = sum(y_true_label)
    per_label_support[label] = support

    if support > 0 or sum(y_pred_label) > 0:
        try:
            label_precision = precision_score(y_true_label, y_pred_label, average='binary', zero_division=0, pos_label=1)
            label_recall = recall_score(y_true_label, y_pred_label, average='binary', zero_division=0, pos_label=1)
            label_f1 = f1_score(y_true_label, y_pred_label, average='binary', zero_division=0, pos_label=1)

            per_label_precision[label] = float(label_precision)
            per_label_recall[label] = float(label_recall)
            per_label_f1[label] = float(label_f1)

            print(f"  {label:15s} - F1: {label_f1:.4f}, P: {label_precision:.4f}, R: {label_recall:.4f}, Support: {support}")
        except Exception as e:
            print(f"  ⚠️  Error for '{label}': {e}")
            per_label_precision[label] = 0.0
            per_label_recall[label] = 0.0
            per_label_f1[label] = 0.0
    else:
        per_label_precision[label] = 0.0
        per_label_recall[label] = 0.0
        per_label_f1[label] = 0.0
        print(f"  {label:15s} - No samples")

baseline_metrics['per_label_f1'] = per_label_f1
baseline_metrics['per_label_precision'] = per_label_precision
baseline_metrics['per_label_recall'] = per_label_recall
baseline_metrics['per_label_support'] = per_label_support
print()


# ========== CALCULATE NLP METRICS (BLEU & ROUGE) ==========
print("="*70)
print("STEP 5: CALCULATING NLP METRICS (BLEU & ROUGE)")
print("="*70)

try:
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    from rouge_score import rouge_scorer
    import nltk

    # Download punkt if needed
    try:
        nltk.data.find('tokenizers/punkt')
    except LookupError:
        print("Downloading NLTK punkt tokenizer...")
        nltk.download('punkt', quiet=True)

    from nltk.tokenize import word_tokenize

    rouge_scorer_obj = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    bleu_scores, rouge1_scores, rouge2_scores, rougeL_scores = [], [], [], []
    smoothing = SmoothingFunction().method1

    for pred_item in tqdm(baseline_preds, desc="Calculating NLP metrics"):
        gt = pred_item.get('ground_truth', '')
        pr = pred_item.get('prediction', '')

        if gt and pr and isinstance(gt, str) and isinstance(pr, str):
            try:
                # BLEU Score
                reference = [word_tokenize(gt.lower())]
                hypothesis = word_tokenize(pr.lower())

                if len(hypothesis) > 0:
                    bleu = sentence_bleu(reference, hypothesis, smoothing_function=smoothing)
                    bleu_scores.append(bleu)
                else:
                    bleu_scores.append(0.0)

                # ROUGE Scores
                rouge_result = rouge_scorer_obj.score(gt, pr)
                rouge1_scores.append(rouge_result['rouge1'].fmeasure)
                rouge2_scores.append(rouge_result['rouge2'].fmeasure)
                rougeL_scores.append(rouge_result['rougeL'].fmeasure)

            except Exception as e:
                bleu_scores.append(0.0)
                rouge1_scores.append(0.0)
                rouge2_scores.append(0.0)
                rougeL_scores.append(0.0)
        else:
            bleu_scores.append(0.0)
            rouge1_scores.append(0.0)
            rouge2_scores.append(0.0)
            rougeL_scores.append(0.0)

    # Add NLP metrics to results
    baseline_metrics.update({
        "bleu_score": float(np.mean(bleu_scores)) if bleu_scores else 0.0,
        "rouge1_score": float(np.mean(rouge1_scores)) if rouge1_scores else 0.0,
        "rouge2_score": float(np.mean(rouge2_scores)) if rouge2_scores else 0.0,
        "rougeL_score": float(np.mean(rougeL_scores)) if rougeL_scores else 0.0,
        "bleu_std": float(np.std(bleu_scores)) if bleu_scores else 0.0,
        "rouge1_std": float(np.std(rouge1_scores)) if rouge1_scores else 0.0,
        "rouge2_std": float(np.std(rouge2_scores)) if rouge2_scores else 0.0,
        "rougeL_std": float(np.std(rougeL_scores)) if rougeL_scores else 0.0,
    })

    print(f"✓ BLEU Score:           {baseline_metrics['bleu_score']:.4f} (±{baseline_metrics['bleu_std']:.4f})")
    print(f"✓ ROUGE-1:              {baseline_metrics['rouge1_score']:.4f} (±{baseline_metrics['rouge1_std']:.4f})")
    print(f"✓ ROUGE-2:              {baseline_metrics['rouge2_score']:.4f} (±{baseline_metrics['rouge2_std']:.4f})")
    print(f"✓ ROUGE-L:              {baseline_metrics['rougeL_score']:.4f} (±{baseline_metrics['rougeL_std']:.4f})\n")

    nlp_available = True

except ImportError as e:
    print(f"⚠️  NLTK/ROUGE packages not available")
    print(f"   Install with: !pip install nltk rouge-score")
    baseline_metrics.update({
        "bleu_score": None,
        "rouge1_score": None,
        "rouge2_score": None,
        "rougeL_score": None,
    })
    nlp_available = False
    print()


# ========== CALCULATE LABEL DISTRIBUTION ==========
print("="*70)
print("STEP 6: CALCULATING LABEL DISTRIBUTION")
print("="*70)

gt_distribution = Counter(y_true)
pred_distribution = Counter(y_pred)

baseline_metrics['ground_truth_distribution'] = dict(gt_distribution)
baseline_metrics['prediction_distribution'] = dict(pred_distribution)

print("Ground Truth Distribution:")
for label, count in gt_distribution.most_common():
    print(f"  {label:15s}: {count:4d} ({count/len(y_true)*100:.1f}%)")

print("\nPrediction Distribution:")
for label, count in pred_distribution.most_common():
    print(f"  {label:15s}: {count:4d} ({count/len(y_pred)*100:.1f}%)")
print()


# ========== GENERATE CLASSIFICATION REPORT ==========
print("="*70)
print("STEP 7: GENERATING CLASSIFICATION REPORT")
print("="*70)

class_report = classification_report(y_true, y_pred, labels=labels, zero_division=0)
print(class_report)

baseline_metrics['classification_report'] = class_report


# ========== SAVE METRICS ==========
print("="*70)
print("STEP 8: SAVING METRICS")
print("="*70)

with open(BASELINE_METRICS_JSON, "w", encoding="utf-8") as f:
    json.dump(baseline_metrics, f, indent=2, ensure_ascii=False)
print(f"✓ Saved metrics to: {BASELINE_METRICS_JSON.name}")

# Save evaluation report (JSON format)
evaluation_report = {
    "model": "AISG SEA-LION v4 32B IT (Baseline)",
    "evaluation_date": pd.Timestamp.now().isoformat(),
    "metrics": baseline_metrics,
    "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
}

with open(BASELINE_REPORT_JSON, "w", encoding="utf-8") as f:
    json.dump(evaluation_report, f, indent=2, ensure_ascii=False)
print(f"✓ Saved evaluation report to: {BASELINE_REPORT_JSON.name}")

# Save evaluation report (TXT format)
with open(BASELINE_REPORT_TXT, "w", encoding="utf-8") as f:
    f.write("="*70 + "\n")
    f.write("PropInsight Baseline SEA-LION Evaluation Report\n")
    f.write("="*70 + "\n\n")
    f.write(f"Model: AISG SEA-LION v4 32B IT (Baseline)\n")
    f.write(f"Evaluation Date: {pd.Timestamp.now()}\n")
    f.write(f"Total Samples: {baseline_metrics['total_samples']}\n\n")

    f.write("-"*70 + "\n")
    f.write("OVERALL METRICS\n")
    f.write("-"*70 + "\n")
    f.write(f"Accuracy:          {baseline_metrics['overall_sentiment_accuracy']:.4f}\n")
    f.write(f"Macro Precision:   {baseline_metrics['macro_precision']:.4f}\n")
    f.write(f"Macro Recall:      {baseline_metrics['macro_recall']:.4f}\n")
    f.write(f"Macro F1:          {baseline_metrics['macro_f1']:.4f}\n")
    f.write(f"Weighted F1:       {baseline_metrics['weighted_f1']:.4f}\n\n")

    if baseline_metrics.get('bleu_score') is not None:
        f.write("-"*70 + "\n")
        f.write("NLP GENERATION METRICS\n")
        f.write("-"*70 + "\n")
        f.write(f"BLEU Score:        {baseline_metrics['bleu_score']:.4f}\n")
        f.write(f"ROUGE-1:           {baseline_metrics['rouge1_score']:.4f}\n")
        f.write(f"ROUGE-2:           {baseline_metrics['rouge2_score']:.4f}\n")
        f.write(f"ROUGE-L:           {baseline_metrics['rougeL_score']:.4f}\n\n")

    f.write("-"*70 + "\n")
    f.write("CLASSIFICATION REPORT\n")
    f.write("-"*70 + "\n")
    f.write(class_report + "\n\n")

    f.write("-"*70 + "\n")
    f.write("PER-LABEL F1 SCORES\n")
    f.write("-"*70 + "\n")
    for label in sorted(per_label_f1.keys(), key=lambda x: per_label_f1[x], reverse=True):
        f.write(f"{label:20s}: {per_label_f1[label]:.4f} (Support: {per_label_support[label]})\n")

print(f"✓ Saved text report to: {BASELINE_REPORT_TXT.name}\n")


# ========== VISUALIZATION 1: COMPREHENSIVE DASHBOARD ==========
print("="*70)
print("STEP 9: GENERATING VISUALIZATIONS")
print("="*70)
print("Creating comprehensive dashboard...")

fig = plt.figure(figsize=(20, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# Set style
sns.set_style("whitegrid")
plt.rcParams['font.size'] = 10

# Plot 1: Overall Metrics Bar Chart (Top Left)
ax1 = fig.add_subplot(gs[0, 0])
metrics_names = ['Accuracy', 'Macro F1', 'Weighted F1', 'Macro Precision', 'Macro Recall']
metrics_values = [
    baseline_metrics['overall_sentiment_accuracy'],
    baseline_metrics['macro_f1'],
    baseline_metrics['weighted_f1'],
    baseline_metrics['macro_precision'],
    baseline_metrics['macro_recall']
]
colors_metrics = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']
bars = ax1.barh(metrics_names, metrics_values, color=colors_metrics, alpha=0.8, edgecolor='black', linewidth=1)
for bar, val in zip(bars, metrics_values):
    ax1.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.3f}',
            va='center', fontsize=9, fontweight='bold')
ax1.set_xlim(0, 1.1)
ax1.set_xlabel('Score', fontweight='bold')
ax1.set_title('Overall Classification Metrics', fontsize=12, fontweight='bold', pad=10)
ax1.grid(True, alpha=0.3, axis='x', linestyle='--')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Plot 2: NLP Metrics (Top Middle)
ax2 = fig.add_subplot(gs[0, 1])
if nlp_available and baseline_metrics.get('bleu_score') is not None:
    nlp_names = ['BLEU', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L']
    nlp_values = [
        baseline_metrics['bleu_score'],
        baseline_metrics['rouge1_score'],
        baseline_metrics['rouge2_score'],
        baseline_metrics['rougeL_score']
    ]
    colors_nlp = ['#3498db', '#e74c3c', '#f39c12', '#9b59b6']
    bars = ax2.bar(nlp_names, nlp_values, color=colors_nlp, alpha=0.8, edgecolor='black', linewidth=1)
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax2.set_ylim(0, 1.1)
    ax2.set_ylabel('Score', fontweight='bold')
    ax2.set_title('NLP Generation Metrics', fontsize=12, fontweight='bold', pad=10)
    ax2.grid(True, alpha=0.3, axis='y', linestyle='--')
else:
    ax2.text(0.5, 0.5, 'NLP metrics\nnot available', ha='center', va='center', fontsize=11)
    ax2.set_title('NLP Generation Metrics', fontsize=12, fontweight='bold', pad=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Plot 3: Label Distribution (Top Right)
ax3 = fig.add_subplot(gs[0, 2])
x = np.arange(len(labels))
width = 0.35
gt_vals = [gt_distribution.get(l, 0) for l in labels]
pred_vals = [pred_distribution.get(l, 0) for l in labels]
bars1 = ax3.bar(x - width/2, gt_vals, width, label='Ground Truth', color='#3498db', alpha=0.8, edgecolor='black')
bars2 = ax3.bar(x + width/2, pred_vals, width, label='Predicted', color='#27ae60', alpha=0.8, edgecolor='black')
ax3.set_xlabel('Sentiment', fontweight='bold')
ax3.set_ylabel('Count', fontweight='bold')
ax3.set_title('Sentiment Distribution', fontsize=12, fontweight='bold', pad=10)
ax3.set_xticks(x)
ax3.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3, axis='y', linestyle='--')
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

# Plot 4: Confusion Matrix (Middle Row, spans 2 columns)
ax4 = fig.add_subplot(gs[1, :2])
cm = confusion_matrix(y_true, y_pred, labels=labels)
with np.errstate(invalid='ignore', divide='ignore'):
    cm_normalized = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
    cm_normalized = np.nan_to_num(cm_normalized)
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=ax4,
            cbar_kws={"label": "Proportion"}, linewidths=0.5,
            linecolor='gray', square=True, cbar=True)
ax4.set_xlabel('Predicted', fontweight='bold')
ax4.set_ylabel('Ground Truth', fontweight='bold')
ax4.set_title('Confusion Matrix (Normalized)', fontsize=12, fontweight='bold', pad=10)

# Plot 5: Top 10 Per-Label F1 (Middle Right)
ax5 = fig.add_subplot(gs[1, 2])
if per_label_f1:
    sorted_items = sorted(per_label_f1.items(), key=lambda x: x[1], reverse=True)[:10]
    label_names = [item[0] for item in sorted_items]
    f1_scores = [item[1] for item in sorted_items]
    colors_f1 = ['#27ae60' if f1 >= 0.7 else '#f39c12' if f1 >= 0.5 else '#e74c3c' for f1 in f1_scores]
    bars = ax5.barh(label_names, f1_scores, color=colors_f1, alpha=0.8, edgecolor='black', linewidth=1)
    for i, (bar, val) in enumerate(zip(bars, f1_scores)):
        ax5.text(val + 0.02, i, f'{val:.2f}', va='center', fontsize=8, fontweight='bold')
    ax5.set_xlabel('F1 Score', fontweight='bold')
    ax5.set_title('Top 10 Per-Label F1', fontsize=12, fontweight='bold', pad=10)
    ax5.set_xlim(0, 1.1)
    ax5.grid(True, alpha=0.3, axis='x', linestyle='--')
    ax5.invert_yaxis()
else:
    ax5.text(0.5, 0.5, "No data", ha='center', va='center', fontsize=11)
    ax5.set_title('Top 10 Per-Label F1', fontsize=12, fontweight='bold')
ax5.spines['top'].set_visible(False)
ax5.spines['right'].set_visible(False)

# Plot 6: All Per-Label F1 Scores (Bottom Row, full width)
ax6 = fig.add_subplot(gs[2, :])
if per_label_f1:
    sorted_items = sorted(per_label_f1.items(), key=lambda x: x[1], reverse=True)
    label_names = [item[0] for item in sorted_items]
    f1_scores = [item[1] for item in sorted_items]
    supports = [per_label_support.get(item[0], 0) for item in sorted_items]

    colors_f1 = ['#27ae60' if f1 >= 0.7 else '#f39c12' if f1 >= 0.5 else '#e74c3c' for f1 in f1_scores]
    bars = ax6.barh(label_names, f1_scores, color=colors_f1, alpha=0.8, edgecolor='black', linewidth=0.8)

    for i, (bar, f1, supp) in enumerate(zip(bars, f1_scores, supports)):
        ax6.text(f1 + 0.02, i, f'{f1:.3f} (n={supp})', va='center', fontsize=7, fontweight='bold')

    ax6.set_xlabel('F1 Score', fontweight='bold', fontsize=11)
    ax6.set_ylabel('Sentiment Label', fontweight='bold', fontsize=11)
    ax6.set_title('All Per-Label F1 Scores with Support', fontsize=12, fontweight='bold', pad=10)
    ax6.set_xlim(0, 1.15)
    ax6.grid(True, alpha=0.3, axis='x', linestyle='--')
    ax6.invert_yaxis()

    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#27ae60', label='Good (F1 ≥ 0.7)'),
        Patch(facecolor='#f39c12', label='Fair (0.5 ≤ F1 < 0.7)'),
        Patch(facecolor='#e74c3c', label='Poor (F1 < 0.5)')
    ]
    ax6.legend(handles=legend_elements, loc='lower right', fontsize=9)
ax6.spines['top'].set_visible(False)
ax6.spines['right'].set_visible(False)

# Add main title
fig.suptitle('PropInsight Baseline SEA-LION - Comprehensive Evaluation Dashboard',
             fontsize=16, fontweight='bold', y=0.995)

plt.savefig(VIS_COMPREHENSIVE, dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Saved comprehensive dashboard to: {VIS_COMPREHENSIVE.name}")


# ========== ADDITIONAL INDIVIDUAL VISUALIZATIONS ==========
# Sentiment Comparison
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(labels))
width = 0.35
gt_vals = [gt_distribution.get(l, 0) for l in labels]
pred_vals = [pred_distribution.get(l, 0) for l in labels]
bars1 = ax.bar(x - width/2, gt_vals, width, label='Ground Truth', color='#3498db', alpha=0.8, edgecolor='black', linewidth=1)
bars2 = ax.bar(x + width/2, pred_vals, width, label='Predicted', color='#27ae60', alpha=0.8, edgecolor='black', linewidth=1)
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width()/2., height, f'{int(height)}', ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Sentiment', fontsize=12, fontweight='bold')
ax.set_ylabel('Count', fontsize=12, fontweight='bold')
ax.set_title('Overall Sentiment Distribution — Baseline SEA-LION (GT vs Pred)', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(VIS_SENTIMENT_COMPARISON, dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Saved sentiment comparison to: {VIS_SENTIMENT_COMPARISON.name}")

# Confusion Matrix (standalone)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=ax,
            cbar_kws={"label": "Proportion"}, linewidths=0.5,
            linecolor='gray', square=True)
ax.set_xlabel('Predicted Sentiment', fontsize=12, fontweight='bold')
ax.set_ylabel('Ground Truth Sentiment', fontsize=12, fontweight='bold')
ax.set_title('Overall Sentiment — Baseline SEA-LION Confusion Matrix (Normalized)', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(VIS_CONFUSION_MATRIX, dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Saved confusion matrix to: {VIS_CONFUSION_MATRIX.name}")

# Per-Label F1 (standalone)
if per_label_f1:
    fig, ax = plt.subplots(figsize=(12, max(8, len(per_label_f1) * 0.4)))
    sorted_items = sorted(per_label_f1.items(), key=lambda x: x[1], reverse=True)
    label_names = [item[0] for item in sorted_items]
    f1_scores = [item[1] for item in sorted_items]
    colors_f1 = ['#27ae60' if f1 >= 0.7 else '#f39c12' if f1 >= 0.5 else '#e74c3c' for f1 in f1_scores]
    bars = ax.barh(label_names, f1_scores, color=colors_f1, alpha=0.8, edgecolor='black', linewidth=1)
    for i, (bar, f1, label) in enumerate(zip(bars, f1_scores, label_names)):
        support = per_label_support.get(label, 0)
        ax.text(f1 + 0.02, i, f'{f1:.3f} (n={support})', va='center', fontsize=9, fontweight='bold')
    ax.set_xlabel('F1 Score', fontsize=12, fontweight='bold')
    ax.set_ylabel('Sentiment Label', fontsize=12, fontweight='bold')
    ax.set_title('Per-Sentiment F1 Scores — Baseline SEA-LION', fontsize=14, fontweight='bold', pad=15)
    ax.set_xlim(0, 1.1)
    ax.grid(True, alpha=0.3, axis='x', linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.invert_yaxis()
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#27ae60', label='Good (F1 ≥ 0.7)'),
        Patch(facecolor='#f39c12', label='Fair (0.5 ≤ F1 < 0.7)'),
        Patch(facecolor='#e74c3c', label='Poor (F1 < 0.5)')
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
    plt.tight_layout()
    plt.savefig(VIS_PER_LABEL_F1, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved per-label F1 to: {VIS_PER_LABEL_F1.name}")

# NLP Metrics (standalone)
if nlp_available and baseline_metrics.get('bleu_score') is not None:
    fig, ax = plt.subplots(figsize=(10, 6))
    nlp_names = ['BLEU', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L']
    nlp_values = [
        baseline_metrics['bleu_score'],
        baseline_metrics['rouge1_score'],
        baseline_metrics['rouge2_score'],
        baseline_metrics['rougeL_score']
    ]
    colors_nlp = ['#3498db', '#e74c3c', '#f39c12', '#9b59b6']
    bars = ax.bar(nlp_names, nlp_values, color=colors_nlp, alpha=0.8, edgecolor='black', linewidth=1.2)
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{height:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax.set_title('NLP Generation Metrics (Baseline SEA-LION)', fontsize=14, fontweight='bold', pad=15)
    ax.set_ylim(0, 1.1)
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig(VIS_NLP_METRICS, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved NLP metrics to: {VIS_NLP_METRICS.name}")


# ========== SUMMARY ==========
print("\n" + "="*70)
print("EVALUATION COMPLETE!")
print("="*70)
print(f"\n📊 Key Metrics Summary:")
print(f"   • Total Samples:        {baseline_metrics['total_samples']}")
print(f"   • Overall Accuracy:     {baseline_metrics['overall_sentiment_accuracy']:.4f}")
print(f"   • Macro F1 Score:       {baseline_metrics['macro_f1']:.4f}")
print(f"   • Weighted F1 Score:    {baseline_metrics['weighted_f1']:.4f}")

if baseline_metrics.get('bleu_score') is not None:
    print(f"   • BLEU Score:           {baseline_metrics['bleu_score']:.4f}")
    print(f"   • ROUGE-L Score:        {baseline_metrics['rougeL_score']:.4f}")

print(f"\n📁 Output Files:")
print(f"   • Metrics (JSON):       {BASELINE_METRICS_JSON.name}")
print(f"   • Report (JSON):        {BASELINE_REPORT_JSON.name}")
print(f"   • Report (TXT):         {BASELINE_REPORT_TXT.name}")

print(f"\n📈 Visualizations ({len([f for f in [VIS_COMPREHENSIVE, VIS_SENTIMENT_COMPARISON, VIS_CONFUSION_MATRIX, VIS_PER_LABEL_F1, VIS_NLP_METRICS] if f.exists()])} files):")
if VIS_COMPREHENSIVE.exists():
    print(f"   • Comprehensive:        {VIS_COMPREHENSIVE.name}")
if VIS_SENTIMENT_COMPARISON.exists():
    print(f"   • Sentiment Dist:       {VIS_SENTIMENT_COMPARISON.name}")
if VIS_CONFUSION_MATRIX.exists():
    print(f"   • Confusion Matrix:     {VIS_CONFUSION_MATRIX.name}")
if VIS_PER_LABEL_F1.exists():
    print(f"   • Per-Label F1:         {VIS_PER_LABEL_F1.name}")
if VIS_NLP_METRICS.exists():
    print(f"   • NLP Metrics:          {VIS_NLP_METRICS.name}")

print(f"\n📂 All files saved to:")
print(f"   • Results:     {RESULTS_DIR}")
print(f"   • Visuals:     {VISUALIZATIONS_DIR}")

print("\n✅ Baseline SEA-LION evaluation complete!")
print("="*70)

STEP 3: CALCULATING STANDARD METRICS
✓ Overall Accuracy:     0.6534
✓ Macro Precision:      0.5507
✓ Macro Recall:         0.3414
✓ Macro F1:             0.2804
✓ Weighted F1:          0.5250

STEP 4: CALCULATING PER-LABEL METRICS
  negative        - F1: 0.0526, P: 1.0000, R: 0.0270, Support: 148
  neutral         - F1: 0.7885, P: 0.6521, R: 0.9972, Support: 359
  positive        - F1: 0.0000, P: 0.0000, R: 0.0000, Support: 47

STEP 5: CALCULATING NLP METRICS (BLEU & ROUGE)
⚠️  NLTK/ROUGE packages not available
   Install with: !pip install nltk rouge-score

STEP 6: CALCULATING LABEL DISTRIBUTION
Ground Truth Distribution:
  neutral        :  359 (64.8%)
  negative       :  148 (26.7%)
  positive       :   47 (8.5%)

Prediction Distribution:
  neutral        :  549 (99.1%)
  negative       :    4 (0.7%)
  positive       :    1 (0.2%)

STEP 7: GENERATING CLASSIFICATION REPORT
              precision    recall  f1-score   support

    negative       1.00      0.03      0.05       148
   

In [8]:
# ========== UPLOAD FINETUNED PREDICTIONS ==========
print("="*70)
print("UPLOAD FINETUNED PREDICTIONS")
print("="*70)

from google.colab import files
from pathlib import Path

BASE = Path("/content/drive/MyDrive/PropInsight")
RESULTS_DIR = BASE / "results"
FINETUNED_PREDICTIONS_FILENAME = "finetuned_predictions_faster.json" # Updated filename
FINETUNED_PREDICTIONS_PATH = "/content/finetuned_predictions_faster.json"

print(f"Please upload the finetuned predictions file: {FINETUNED_PREDICTIONS_FILENAME}")

try:
    uploaded = files.upload()

    if FINETUNED_PREDICTIONS_FILENAME in uploaded:
        # Save the uploaded file to the correct location
        try:
            with open(FINETUNED_PREDICTIONS_PATH, 'wb') as f:
                f.write(uploaded[FINETUNED_PREDICTIONS_FILENAME])
            print(f"✓ Successfully uploaded and saved {FINETUNED_PREDICTIONS_FILENAME} to {FINETUNED_PREDICTIONS_PATH}")
        except Exception as e:
            print(f"❌ Error saving uploaded file: {e}")
            print(f"Uploaded file is temporarily available in the current directory.")
            raise # Re-raise the exception
    else:
        print("❌ Upload cancelled or incorrect file name.")
        raise FileNotFoundError(f"Finetuned prediction file not found after upload attempt: {FINETUNED_PREDICTIONS_FILENAME}")

except Exception as e:
    print(f"An error occurred during the upload process: {e}")
    raise

UPLOAD FINETUNED PREDICTIONS
Please upload the finetuned predictions file: finetuned_predictions_faster.json


Saving finetuned_predictions_faster.json to finetuned_predictions_faster.json
✓ Successfully uploaded and saved finetuned_predictions_faster.json to /content/finetuned_predictions_faster.json


In [13]:
# ============================================================
# PropInsight — Finetuned SEA-LION Comprehensive Evaluation
# ============================================================

import os, json, re, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
from collections import Counter
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

print("="*70)
print("PropInsight - Finetuned SEA-LION Comprehensive Evaluation")
print("="*70 + "\n")

# ========== CONFIGURATION ==========
BASE = Path("/content/drive/MyDrive/PropInsight")
RESULTS_DIR = BASE / "results"
VISUALIZATIONS_DIR = BASE / "visualizations"
FINETUNED_PREDICTIONS = "/content/finetuned_predictions_faster.json" # This was set in the previous upload cell

# Output files with _sealion_finetune suffix
FINETUNED_METRICS_JSON = RESULTS_DIR / "finetuned_Sealion_metrics.json"
FINETUNED_REPORT_JSON = RESULTS_DIR / "finetuned_Sealion_evaluation_report.json"
FINETUNED_REPORT_TXT = RESULTS_DIR / "finetuned_Sealion_evaluation_report.txt"

# Visualization outputs with _sealion_finetune suffix
VIS_NLP_METRICS_FINETUNED = VISUALIZATIONS_DIR / "finetuned_Sealion_nlp_metrics.png"
VIS_SENTIMENT_COMPARISON_FINETUNED = VISUALIZATIONS_DIR / "finetuned_Sealion_sentiment_comparison.png"
VIS_CONFUSION_MATRIX_FINETUNED = VISUALIZATIONS_DIR / "finetuned_Sealion_confusion_matrix.png"
VIS_PER_LABEL_F1_FINETUNED = VISUALIZATIONS_DIR / "finetuned_Sealion_per_label_f1.png"
VIS_COMPREHENSIVE_FINETUNED = VISUALIZATIONS_DIR / "finetuned_Sealion_comprehensive_metrics.png"


# Ensure directories exist
for d in [RESULTS_DIR, VISUALIZATIONS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"✓ Base directory: {BASE}")
print(f"✓ Loading predictions from: {FINETUNED_PREDICTIONS}\n") # Removed .name


# ========== HELPER FUNCTIONS ==========
def extract_field(block: str, key: str, default: str = "") -> str:
    """Extract a field value from the output block."""
    if not isinstance(block, str):
        return default
    m = re.search(rf"{re.escape(key)}\s*:\s*(.*?)(?:\n|$)", block, flags=re.IGNORECASE)
    return (m.group(1).strip() if m else default)


def normalize_sentiment(sentiment: str) -> str:
    """Normalize sentiment labels to lowercase and handle variations."""
    if not isinstance(sentiment, str):
        return "neutral"

    sentiment = sentiment.lower().strip()

    # Handle common variations
    if sentiment in ["positive", "pos"]:
        return "positive"
    elif sentiment in ["negative", "neg"]:
        return "negative"
    elif sentiment in ["neutral", "neu", "mixed"]:
        return "neutral"
    else:
        return sentiment if sentiment else "neutral"

# ========== LOAD PREDICTIONS ==========
print("="*70)
print("STEP 1: LOADING FINETUNED PREDICTIONS")
print("="*70)

try:
    with open(FINETUNED_PREDICTIONS, "r", encoding="utf-8") as f:
        finetuned_preds = json.load(f)
    print(f"✓ Successfully loaded {len(finetuned_preds)} predictions\n")
except FileNotFoundError:
    print(f"❌ File not found: {FINETUNED_PREDICTIONS}")
    print("Please ensure finetuned predictions have been uploaded or generated.")
    raise # Re-raise the exception to stop execution if the file is missing
except Exception as e:
    print(f"❌ ERROR loading finetuned predictions: {e}")
    raise


# ========== EXTRACT OVERALL SENTIMENT ==========
print("="*70)
print("STEP 2: EXTRACTING OVERALL SENTIMENT")
print("="*70)

y_true = []
y_pred = []

for item in finetuned_preds:
    gt = extract_field(item.get("ground_truth", ""), "Overall Sentiment", "neutral")
    pred = extract_field(item.get("prediction", ""), "Overall Sentiment", "neutral")

    y_true.append(normalize_sentiment(gt))
    y_pred.append(normalize_sentiment(pred))

print(f"✓ Extracted {len(y_true)} ground truth labels")
print(f"✓ Extracted {len(y_pred)} predictions")

# Get all unique labels from both ground truth and predictions
labels = sorted(list(set(y_true) | set(y_pred)))
print(f"✓ Found {len(labels)} unique sentiment labels: {labels}\n")


# ========== CALCULATE STANDARD METRICS ==========
print("="*70)
print("STEP 3: CALCULATING STANDARD METRICS")
print("="*70)

finetuned_metrics = {
    "model": "AISG SEA-LION v4 32B IT (Finetuned)",
    "total_samples": len(finetuned_preds),
    "labels": labels,
    "overall_sentiment_accuracy": float(accuracy_score(y_true, y_pred)),
    "macro_precision": float(precision_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)),
    "macro_recall": float(recall_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)),
    "macro_f1": float(f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)),
    "weighted_precision": float(precision_score(y_true, y_pred, labels=labels, average='weighted', zero_division=0)),
    "weighted_recall": float(recall_score(y_true, y_pred, labels=labels, average='weighted', zero_division=0)),
    "weighted_f1": float(f1_score(y_true, y_pred, labels=labels, average='weighted', zero_division=0)),
}

print(f"✓ Overall Accuracy:     {finetuned_metrics['overall_sentiment_accuracy']:.4f}")
print(f"✓ Macro Precision:      {finetuned_metrics['macro_precision']:.4f}")
print(f"✓ Macro Recall:         {finetuned_metrics['macro_recall']:.4f}")
print(f"✓ Macro F1:             {finetuned_metrics['macro_f1']:.4f}")
print(f"✓ Weighted F1:          {finetuned_metrics['weighted_f1']:.4f}\n")


# ========== CALCULATE PER-LABEL METRICS ==========
print("="*70)
print("STEP 4: CALCULATING PER-LABEL METRICS")
print("="*70)

per_label_f1 = {}
per_label_precision = {}
per_label_recall = {}
per_label_support = {}

for label in labels:
    # Binary classification for each label
    y_true_label = [1 if yt == label else 0 for yt in y_true]
    y_pred_label = [1 if yp == label else 0 for yp in y_pred]

    # Count support (number of true instances)
    support = sum(y_true_label)
    per_label_support[label] = support

    if support > 0 or sum(y_pred_label) > 0:
        try:
            label_precision = precision_score(y_true_label, y_pred_label, average='binary', zero_division=0, pos_label=1)
            label_recall = recall_score(y_true_label, y_pred_label, average='binary', zero_division=0, pos_label=1)
            label_f1 = f1_score(y_true_label, y_pred_label, average='binary', zero_division=0, pos_label=1)

            per_label_precision[label] = float(label_precision)
            per_label_recall[label] = float(label_recall)
            per_label_f1[label] = float(label_f1)

            print(f"  {label:15s} - F1: {label_f1:.4f}, P: {label_precision:.4f}, R: {label_recall:.4f}, Support: {support}")
        except Exception as e:
            print(f"  ⚠️  Error for '{label}': {e}")
            per_label_precision[label] = 0.0
            per_label_recall[label] = 0.0
            per_label_f1[label] = 0.0
    else:
        per_label_precision[label] = 0.0
        per_label_recall[label] = 0.0
        per_label_f1[label] = 0.0
        print(f"  {label:15s} - No samples")

finetuned_metrics['per_label_f1'] = per_label_f1
finetuned_metrics['per_label_precision'] = per_label_precision
finetuned_metrics['per_label_recall'] = per_label_recall
finetuned_metrics['per_label_support'] = per_label_support
print()


# ========== CALCULATE NLP METRICS (BLEU & ROUGE) ==========
print("="*70)
print("STEP 5: CALCULATING NLP METRICS (BLEU & ROUGE)")
print("="*70)

try:
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    from rouge_score import rouge_scorer
    import nltk

    # Download punkt if needed
    try:
        nltk.data.find('tokenizers/punkt')
    except LookupError:
        print("Downloading NLTK punkt tokenizer...")
        nltk.download('punkt', quiet=True)

    from nltk.tokenize import word_tokenize

    rouge_scorer_obj = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    bleu_scores, rouge1_scores, rouge2_scores, rougeL_scores = [], [], [], []
    smoothing = SmoothingFunction().method1

    for pred_item in tqdm(finetuned_preds, desc="Calculating NLP metrics"):
        gt = pred_item.get('ground_truth', '')
        pr = pred_item.get('prediction', '')

        if gt and pr and isinstance(gt, str) and isinstance(pr, str):
            try:
                # BLEU Score
                reference = [word_tokenize(gt.lower())]
                hypothesis = word_tokenize(pr.lower())

                if len(hypothesis) > 0:
                    bleu = sentence_bleu(reference, hypothesis, smoothing_function=smoothing)
                    bleu_scores.append(bleu)
                else:
                    bleu_scores.append(0.0)

                # ROUGE Scores
                rouge_result = rouge_scorer_obj.score(gt, pr)
                rouge1_scores.append(rouge_result['rouge1'].fmeasure)
                rouge2_scores.append(rouge_result['rouge2'].fmeasure)
                rougeL_scores.append(rouge_result['rougeL'].fmeasure)

            except Exception as e:
                bleu_scores.append(0.0)
                rouge1_scores.append(0.0)
                rouge2_scores.append(0.0)
                rougeL_scores.append(0.0)
        else:
            bleu_scores.append(0.0)
            rouge1_scores.append(0.0)
            rouge2_scores.append(0.0)
            rougeL_scores.append(0.0)

    # Add NLP metrics to results
    finetuned_metrics.update({
        "bleu_score": float(np.mean(bleu_scores)) if bleu_scores else 0.0,
        "rouge1_score": float(np.mean(rouge1_scores)) if rouge1_scores else 0.0,
        "rouge2_score": float(np.mean(rouge2_scores)) if rouge2_scores else 0.0,
        "rougeL_score": float(np.mean(rougeL_scores)) if rougeL_scores else 0.0,
        "bleu_std": float(np.std(bleu_scores)) if bleu_scores else 0.0,
        "rouge1_std": float(np.std(rouge1_scores)) if rouge1_scores else 0.0,
        "rouge2_std": float(np.std(rouge2_scores)) if rouge2_scores else 0.0,
        "rougeL_std": float(np.std(rougeL_scores)) if rougeL_scores else 0.0,
    })

    print(f"✓ BLEU Score:           {finetuned_metrics['bleu_score']:.4f} (±{finetuned_metrics['bleu_std']:.4f})")
    print(f"✓ ROUGE-1:              {finetuned_metrics['rouge1_score']:.4f} (±{finetuned_metrics['rouge1_std']:.4f})")
    print(f"✓ ROUGE-2:              {finetuned_metrics['rouge2_score']:.4f} (±{finetuned_metrics['rouge2_std']:.4f})")
    print(f"✓ ROUGE-L:              {finetuned_metrics['rougeL_score']:.4f} (±{finetuned_metrics['rougeL_std']:.4f})\n")

    nlp_available = True

except ImportError as e:
    print(f"⚠️  NLTK/ROUGE packages not available")
    print(f"   Install with: !pip install nltk rouge-score")
    finetuned_metrics.update({
        "bleu_score": None,
        "rouge1_score": None,
        "rouge2_score": None,
        "rougeL_score": None,
    })
    nlp_available = False
    print()


# ========== CALCULATE LABEL DISTRIBUTION ==========
print("="*70)
print("STEP 6: CALCULATING LABEL DISTRIBUTION")
print("="*70)

gt_distribution = Counter(y_true)
pred_distribution = Counter(y_pred)

finetuned_metrics['ground_truth_distribution'] = dict(gt_distribution)
finetuned_metrics['prediction_distribution'] = dict(pred_distribution)

print("Ground Truth Distribution:")
for label, count in gt_distribution.most_common():
    print(f"  {label:15s}: {count:4d} ({count/len(y_true)*100:.1f}%)")

print("\nPrediction Distribution:")
for label, count in pred_distribution.most_common():
    print(f"  {label:15s}: {count:4d} ({count/len(y_pred)*100:.1f}%)")
print()


# ========== GENERATE CLASSIFICATION REPORT ==========
print("="*70)
print("STEP 7: GENERATING CLASSIFICATION REPORT")
print("="*70)

class_report = classification_report(y_true, y_pred, labels=labels, zero_division=0)
print(class_report)

finetuned_metrics['classification_report'] = class_report


# ========== SAVE METRICS ==========
print("="*70)
print("STEP 8: SAVING METRICS")
print("="*70)

with open(FINETUNED_METRICS_JSON, "w", encoding="utf-8") as f:
    json.dump(finetuned_metrics, f, indent=2, ensure_ascii=False)
print(f"✓ Saved metrics to: {FINETUNED_METRICS_JSON.name}")

# Save evaluation report (JSON format)
evaluation_report = {
    "model": "AISG SEA-LION v4 32B IT (Finetuned)",
    "evaluation_date": pd.Timestamp.now().isoformat(),
    "metrics": finetuned_metrics,
    "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
}

with open(FINETUNED_REPORT_JSON, "w", encoding="utf-8") as f:
    json.dump(evaluation_report, f, indent=2, ensure_ascii=False)
print(f"✓ Saved evaluation report to: {FINETUNED_REPORT_JSON.name}")

# Save evaluation report (TXT format)
with open(FINETUNED_REPORT_TXT, "w", encoding="utf-8") as f:
    f.write("="*70 + "\n")
    f.write("PropInsight Finetuned SEA-LION Evaluation Report\n")
    f.write("="*70 + "\n\n")
    f.write(f"Model: AISG SEA-LION v4 32B IT (Finetuned)\n")
    f.write(f"Evaluation Date: {pd.Timestamp.now()}\n")
    f.write(f"Total Samples: {finetuned_metrics['total_samples']}\n\n")

    f.write("-"*70 + "\n")
    f.write("OVERALL METRICS\n")
    f.write("-"*70 + "\n")
    f.write(f"Accuracy:          {finetuned_metrics['overall_sentiment_accuracy']:.4f}\n")
    f.write(f"Macro Precision:   {finetuned_metrics['macro_precision']:.4f}\n")
    f.write(f"Macro Recall:      {finetuned_metrics['macro_recall']:.4f}\n")
    f.write(f"Macro F1:          {finetuned_metrics['macro_f1']:.4f}\n")
    f.write(f"Weighted F1:       {finetuned_metrics['weighted_f1']:.4f}\n\n")

    if finetuned_metrics.get('bleu_score') is not None:
        f.write("-"*70 + "\n")
        f.write("NLP GENERATION METRICS\n")
        f.write("-"*70 + "\n")
        f.write(f"BLEU Score:        {finetuned_metrics['bleu_score']:.4f}\n")
        f.write(f"ROUGE-1:           {finetuned_metrics['rouge1_score']:.4f}\n")
        f.write(f"ROUGE-2:           {finetuned_metrics['rouge2_score']:.4f}\n")
        f.write(f"ROUGE-L:           {finetuned_metrics['rougeL_score']:.4f}\n\n")

    f.write("-"*70 + "\n")
    f.write("CLASSIFICATION REPORT\n")
    f.write("-"*70 + "\n")
    f.write(class_report + "\n\n")

    f.write("-"*70 + "\n")
    f.write("PER-LABEL F1 SCORES\n")
    f.write("-"*70 + "\n")
    for label in sorted(per_label_f1.keys(), key=lambda x: per_label_f1[x], reverse=True):
        f.write(f"{label:20s}: {per_label_f1[label]:.4f} (Support: {per_label_support[label]})\n")

print(f"✓ Saved text report to: {FINETUNED_REPORT_TXT.name}\n")


# ========== VISUALIZATION 1: COMPREHENSIVE DASHBOARD ==========
print("="*70)
print("STEP 9: GENERATING VISUALIZATIONS")
print("="*70)
print("Creating comprehensive dashboard...")

fig = plt.figure(figsize=(20, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# Set style
sns.set_style("whitegrid")
plt.rcParams['font.size'] = 10

# Plot 1: Overall Metrics Bar Chart (Top Left)
ax1 = fig.add_subplot(gs[0, 0])
metrics_names = ['Accuracy', 'Macro F1', 'Weighted F1', 'Macro Precision', 'Macro Recall']
metrics_values = [
    finetuned_metrics['overall_sentiment_accuracy'],
    finetuned_metrics['macro_f1'],
    finetuned_metrics['weighted_f1'],
    finetuned_metrics['macro_precision'],
    finetuned_metrics['macro_recall']
]
colors_metrics = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']
bars = ax1.barh(metrics_names, metrics_values, color=colors_metrics, alpha=0.8, edgecolor='black', linewidth=1)
for bar, val in zip(bars, metrics_values):
    ax1.text(val + 0.01, bar.get_y() + bar.get_height()/2, f'{val:.3f}',
            va='center', fontsize=9, fontweight='bold')
ax1.set_xlim(0, 1.1)
ax1.set_xlabel('Score', fontweight='bold')
ax1.set_title('Overall Classification Metrics', fontsize=12, fontweight='bold', pad=10)
ax1.grid(True, alpha=0.3, axis='x', linestyle='--')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Plot 2: NLP Metrics (Top Middle)
ax2 = fig.add_subplot(gs[0, 1])
if nlp_available and finetuned_metrics.get('bleu_score') is not None:
    nlp_names = ['BLEU', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L']
    nlp_values = [
        finetuned_metrics['bleu_score'],
        finetuned_metrics['rouge1_score'],
        finetuned_metrics['rouge2_score'],
        finetuned_metrics['rougeL_score']
    ]
    colors_nlp = ['#3498db', '#e74c3c', '#f39c12', '#9b59b6']
    bars = ax2.bar(nlp_names, nlp_values, color=colors_nlp, alpha=0.8, edgecolor='black', linewidth=1)
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax2.set_ylim(0, 1.1)
    ax2.set_ylabel('Score', fontweight='bold')
    ax2.set_title('NLP Generation Metrics', fontsize=12, fontweight='bold', pad=10)
    ax2.grid(True, alpha=0.3, axis='y', linestyle='--')
else:
    ax2.text(0.5, 0.5, 'NLP metrics\nnot available', ha='center', va='center', fontsize=11)
    ax2.set_title('NLP Generation Metrics', fontsize=12, fontweight='bold', pad=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Plot 3: Label Distribution (Top Right)
ax3 = fig.add_subplot(gs[0, 2])
x = np.arange(len(labels))
width = 0.35
gt_vals = [gt_distribution.get(l, 0) for l in labels]
pred_vals = [pred_distribution.get(l, 0) for l in labels]
bars1 = ax3.bar(x - width/2, gt_vals, width, label='Ground Truth', color='#3498db', alpha=0.8, edgecolor='black')
bars2 = ax3.bar(x + width/2, pred_vals, width, label='Predicted', color='#27ae60', alpha=0.8, edgecolor='black')
ax3.set_xlabel('Sentiment', fontweight='bold')
ax3.set_ylabel('Count', fontweight='bold')
ax3.set_title('Sentiment Distribution', fontsize=12, fontweight='bold', pad=10)
ax3.set_xticks(x)
ax3.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3, axis='y', linestyle='--')
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

# Plot 4: Confusion Matrix (Middle Row, spans 2 columns)
ax4 = fig.add_subplot(gs[1, :2])
cm = confusion_matrix(y_true, y_pred, labels=labels)
with np.errstate(invalid='ignore', divide='ignore'):
    cm_normalized = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
    cm_normalized = np.nan_to_num(cm_normalized)
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=ax4,
            cbar_kws={"label": "Proportion"}, linewidths=0.5,
            linecolor='gray', square=True, cbar=True)
ax4.set_xlabel('Predicted', fontweight='bold')
ax4.set_ylabel('Ground Truth', fontweight='bold')
ax4.set_title('Confusion Matrix (Normalized)', fontsize=12, fontweight='bold', pad=10)

# Plot 5: Top 10 Per-Label F1 (Middle Right)
ax5 = fig.add_subplot(gs[1, 2])
if per_label_f1:
    sorted_items = sorted(per_label_f1.items(), key=lambda x: x[1], reverse=True)[:10]
    label_names = [item[0] for item in sorted_items]
    f1_scores = [item[1] for item in sorted_items]
    colors_f1 = ['#27ae60' if f1 >= 0.7 else '#f39c12' if f1 >= 0.5 else '#e74c3c' for f1 in f1_scores]
    bars = ax5.barh(label_names, f1_scores, color=colors_f1, alpha=0.8, edgecolor='black', linewidth=1)
    for i, (bar, val) in enumerate(zip(bars, f1_scores)):
        ax5.text(val + 0.02, i, f'{val:.2f}', va='center', fontsize=8, fontweight='bold')
    ax5.set_xlabel('F1 Score', fontweight='bold')
    ax5.set_title('Top 10 Per-Label F1', fontsize=12, fontweight='bold', pad=10)
    ax5.set_xlim(0, 1.1)
    ax5.grid(True, alpha=0.3, axis='x', linestyle='--')
    ax5.invert_yaxis()
else:
    ax5.text(0.5, 0.5, "No data", ha='center', va='center', fontsize=11)
    ax5.set_title('Top 10 Per-Label F1', fontsize=12, fontweight='bold')
ax5.spines['top'].set_visible(False)
ax5.spines['right'].set_visible(False)

# Plot 6: All Per-Label F1 Scores (Bottom Row, full width)
ax6 = fig.add_subplot(gs[2, :])
if per_label_f1:
    sorted_items = sorted(per_label_f1.items(), key=lambda x: x[1], reverse=True)
    label_names = [item[0] for item in sorted_items]
    f1_scores = [item[1] for item in sorted_items]
    supports = [per_label_support.get(item[0], 0) for item in sorted_items]

    colors_f1 = ['#27ae60' if f1 >= 0.7 else '#f39c12' if f1 >= 0.5 else '#e74c3c' for f1 in f1_scores]
    bars = ax6.barh(label_names, f1_scores, color=colors_f1, alpha=0.8, edgecolor='black', linewidth=0.8)

    for i, (bar, f1, supp) in enumerate(zip(bars, f1_scores, supports)):
        ax6.text(f1 + 0.02, i, f'{f1:.3f} (n={supp})', va='center', fontsize=7, fontweight='bold')

    ax6.set_xlabel('F1 Score', fontweight='bold', fontsize=11)
    ax6.set_ylabel('Sentiment Label', fontweight='bold', fontsize=11)
    ax6.set_title('All Per-Label F1 Scores with Support', fontsize=12, fontweight='bold', pad=10)
    ax6.set_xlim(0, 1.15)
    ax6.grid(True, alpha=0.3, axis='x', linestyle='--')
    ax6.invert_yaxis()

    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#27ae60', label='Good (F1 ≥ 0.7)'),
        Patch(facecolor='#f39c12', label='Fair (0.5 ≤ F1 < 0.7)'),
        Patch(facecolor='#e74c3c', label='Poor (F1 < 0.5)')
    ]
    ax6.legend(handles=legend_elements, loc='lower right', fontsize=9)
ax6.spines['top'].set_visible(False)
ax6.spines['right'].set_visible(False)

# Add main title
fig.suptitle('PropInsight Finetuned SEA-LION - Comprehensive Evaluation Dashboard',
             fontsize=16, fontweight='bold', y=0.995)

plt.savefig(VIS_COMPREHENSIVE_FINETUNED, dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Saved comprehensive dashboard to: {VIS_COMPREHENSIVE_FINETUNED.name}")


# ========== ADDITIONAL INDIVIDUAL VISUALIZATIONS ==========
# Sentiment Comparison
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(labels))
width = 0.35
gt_vals = [gt_distribution.get(l, 0) for l in labels]
pred_vals = [pred_distribution.get(l, 0) for l in labels]
bars1 = ax.bar(x - width/2, gt_vals, width, label='Ground Truth', color='#3498db', alpha=0.8, edgecolor='black', linewidth=1)
bars2 = ax.bar(x + width/2, pred_vals, width, label='Predicted', color='#27ae60', alpha=0.8, edgecolor='black', linewidth=1)
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax.text(bar.get_x() + bar.get_width()/2., height, f'{int(height)}', ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Sentiment', fontsize=12, fontweight='bold')
ax.set_ylabel('Count', fontsize=12, fontweight='bold')
ax.set_title('Overall Sentiment Distribution — Finetuned SEA-LION (GT vs Pred)', fontsize=14, fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(VIS_SENTIMENT_COMPARISON_FINETUNED, dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Saved sentiment comparison to: {VIS_SENTIMENT_COMPARISON_FINETUNED.name}")

# Confusion Matrix (standalone)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=ax,
            cbar_kws={"label": "Proportion"}, linewidths=0.5,
            linecolor='gray', square=True)
ax.set_xlabel('Predicted Sentiment', fontsize=12, fontweight='bold')
ax.set_ylabel('Ground Truth Sentiment', fontsize=12, fontweight='bold')
ax.set_title('Overall Sentiment — Finetuned SEA-LION Confusion Matrix (Normalized)', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(VIS_CONFUSION_MATRIX_FINETUNED, dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ Saved confusion matrix to: {VIS_CONFUSION_MATRIX_FINETUNED.name}")

# Per-Label F1 (standalone)
if per_label_f1:
    fig, ax = plt.subplots(figsize=(12, max(8, len(per_label_f1) * 0.4)))
    sorted_items = sorted(per_label_f1.items(), key=lambda x: x[1], reverse=True)
    label_names = [item[0] for item in sorted_items]
    f1_scores = [item[1] for item in sorted_items]
    colors_f1 = ['#27ae60' if f1 >= 0.7 else '#f39c12' if f1 >= 0.5 else '#e74c3c' for f1 in f1_scores]
    bars = ax.barh(label_names, f1_scores, color=colors_f1, alpha=0.8, edgecolor='black', linewidth=1)
    for i, (bar, f1, label) in enumerate(zip(bars, f1_scores, label_names)):
        support = per_label_support.get(label, 0)
        ax.text(f1 + 0.02, i, f'{f1:.3f} (n={support})', va='center', fontsize=9, fontweight='bold')
    ax.set_xlabel('F1 Score', fontsize=12, fontweight='bold')
    ax.set_ylabel('Sentiment Label', fontsize=12, fontweight='bold')
    ax.set_title('Per-Sentiment F1 Scores — Finetuned SEA-LION', fontsize=14, fontweight='bold', pad=15)
    ax.set_xlim(0, 1.1)
    ax.grid(True, alpha=0.3, axis='x', linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.invert_yaxis()
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#27ae60', label='Good (F1 ≥ 0.7)'),
        Patch(facecolor='#f39c12', label='Fair (0.5 ≤ F1 < 0.7)'),
        Patch(facecolor='#e74c3c', label='Poor (F1 < 0.5)')
    ]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=10)
    plt.tight_layout()
    plt.savefig(VIS_PER_LABEL_F1_FINETUNED, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved per-label F1 to: {VIS_PER_LABEL_F1_FINETUNED.name}")

# NLP Metrics (standalone)
if nlp_available and finetuned_metrics.get('bleu_score') is not None:
    fig, ax = plt.subplots(figsize=(10, 6))
    nlp_names = ['BLEU', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L']
    nlp_values = [
        finetuned_metrics['bleu_score'],
        finetuned_metrics['rouge1_score'],
        finetuned_metrics['rouge2_score'],
        finetuned_metrics['rougeL_score']
    ]
    colors_nlp = ['#3498db', '#e74c3c', '#f39c12', '#9b59b6']
    bars = ax.bar(nlp_names, nlp_values, color=colors_nlp, alpha=0.8, edgecolor='black', linewidth=1.2)
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
                f'{height:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax.set_title('NLP Generation Metrics (Finetuned SEA-LION)', fontsize=14, fontweight='bold', pad=15)
    ax.set_ylim(0, 1.1)
    ax.grid(True, alpha=0.3, axis='y', linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig(VIS_NLP_METRICS_FINETUNED, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved NLP metrics to: {VIS_NLP_METRICS_FINETUNED.name}")


# ========== SUMMARY ==========
print("\n" + "="*70)
print("EVALUATION COMPLETE!")
print("="*70)
print(f"\n📊 Key Metrics Summary:")
print(f"   • Total Samples:        {finetuned_metrics['total_samples']}")
print(f"   • Overall Accuracy:     {finetuned_metrics['overall_sentiment_accuracy']:.4f}")
print(f"   • Macro F1 Score:       {finetuned_metrics['macro_f1']:.4f}")
print(f"   • Weighted F1 Score:    {finetuned_metrics['weighted_f1']:.4f}")

if finetuned_metrics.get('bleu_score') is not None:
    print(f"   • BLEU Score:           {finetuned_metrics['bleu_score']:.4f}")
    print(f"   • ROUGE-L Score:        {finetuned_metrics['rougeL_score']:.4f}")

print(f"\n📁 Output Files:")
print(f"   • Metrics (JSON):       {FINETUNED_METRICS_JSON.name}")
print(f"   • Report (JSON):        {FINETUNED_REPORT_JSON.name}")
print(f"   • Report (TXT):         {FINETUNED_REPORT_TXT.name}")

print(f"\n📈 Visualizations ({len([f for f in [VIS_COMPREHENSIVE_FINETUNED, VIS_SENTIMENT_COMPARISON_FINETUNED, VIS_CONFUSION_MATRIX_FINETUNED, VIS_PER_LABEL_F1_FINETUNED, VIS_NLP_METRICS_FINETUNED] if f.exists()])} files):")
if VIS_COMPREHENSIVE_FINETUNED.exists():
    print(f"   • Comprehensive:        {VIS_COMPREHENSIVE_FINETUNED.name}")
if VIS_SENTIMENT_COMPARISON_FINETUNED.exists():
    print(f"   • Sentiment Dist:       {VIS_SENTIMENT_COMPARISON_FINETUNED.name}")
if VIS_CONFUSION_MATRIX_FINETUNED.exists():
    print(f"   • Confusion Matrix:     {VIS_CONFUSION_MATRIX_FINETUNED.name}")
if VIS_PER_LABEL_F1_FINETUNED.exists():
    print(f"   • Per-Label F1:         {VIS_PER_LABEL_F1_FINETUNED.name}")
if VIS_NLP_METRICS_FINETUNED.exists():
    print(f"   • NLP Metrics:          {VIS_NLP_METRICS_FINETUNED.name}")


print(f"\n📂 All files saved to:")
print(f"   • Results:     {RESULTS_DIR}")
print(f"   • Visuals:     {VISUALIZATIONS_DIR}")

print("\n✅ Finetuned SEA-LION evaluation complete!")
print("="*70)

PropInsight - Finetuned SEA-LION Comprehensive Evaluation

✓ Base directory: /content/drive/MyDrive/PropInsight
✓ Loading predictions from: /content/finetuned_predictions_faster.json

STEP 1: LOADING FINETUNED PREDICTIONS
✓ Successfully loaded 554 predictions

STEP 2: EXTRACTING OVERALL SENTIMENT
✓ Extracted 554 ground truth labels
✓ Extracted 554 predictions
✓ Found 4 unique sentiment labels: ['negative', 'neutral', 'positive', 'rising']

STEP 3: CALCULATING STANDARD METRICS
✓ Overall Accuracy:     0.8303
✓ Macro Precision:      0.5854
✓ Macro Recall:         0.5322
✓ Macro F1:             0.5540
✓ Weighted F1:          0.8260

STEP 4: CALCULATING PER-LABEL METRICS
  negative        - F1: 0.7986, P: 0.8538, R: 0.7500, Support: 148
  neutral         - F1: 0.8743, P: 0.8406, R: 0.9109, Support: 359
  positive        - F1: 0.5432, P: 0.6471, R: 0.4681, Support: 47
  rising          - F1: 0.0000, P: 0.0000, R: 0.0000, Support: 0

STEP 5: CALCULATING NLP METRICS (BLEU & ROUGE)
⚠️  NLTK/ROU